In [2]:
import numpy as np
import sys
sys.path.append('/home/mark.bishop/Documents/Kea')
import kea
from kea.utils import binning

In [3]:
from scipy.special import gamma

def volume_hypersphere(
        radius: float,
        dimension: int | float) -> float:
    """volume_hypersphere(radius, dimension)\n
    
    Calculates the volume of a $D$-dimensional hypersphere with radius $r$.

    Args:
        radius (float): Radius of the hypersphere
        dimension (int): (Euclidean) dimension of the hypersphere
    Returns:
        volume (float): Volume
    """
    D = float(dimension)
    return (np.pi**(D/2.) / gamma(D/2. + 1.)) * radius**D

def hypersphere_binshell(
        bins: np.ndarray,
        dimension: int | float,
        bin_widths: np.ndarray,
        grid_widths: tuple,
        centered: Optional[bool] = False) -> np.ndarray:
    """hypersphere_binshell(bins, dimension, bin_widths, grid_widths, centered)\n

    Calculates the number of grid elements within a hyperspherical bin shell for arbitrary dimension $D$.

    Args:
        bins (np.ndarray): Bin array (radius)
        dimension (int|float): Euclidean dimension
        bin_widths (np.ndarray): Bin width size
        grid_widths (tuple): Element widths for each dimension (grid size)
        centered (bool): If true, bins represent the center of the shell, otherwise the inner radius
    """
    if centered:
        r_inner = bins - bin_widths / 2.
        r_outer = bins + bin_widths / 2.
    else:
        r_inner = bins
        r_outer = bins + bin_widths
    
    shell_volume = volume_hypersphere(r_outer, dimension) - volume_hypersphere(r_inner, dimension)
    grid_volume_scale = np.prod(grid_widths)**dimension
    return shell_volume / grid_volume_scale

def kern(bins, D, db, dx):
    """kern(bins, D, db, dx)

    Calculates the number of elements within a bin of width db

    Args:
        bins (np.ndarray): Bin array
        D (int): Euclidean dimension
        db (np.ndarray): Bin width size
        dx (tuple,list): Element widths for each dimension, grid size
    Returns:
        np.ndarray: The number of elements within a bin of width db
    """
    if D == 1:
        return 2.
    elif D == 2:
        return (2. * np.pi * bins * db + np.pi * db**2) / np.prod(dx)**2
    elif D == 3:
        return (4./3.) * np.pi * (3. * bins * db**2 + 3. * bins**2 * db + db**3) / np.prod(dx)**3
    else:
        raise ValueError('Not implemented for dimension %s' % D)

def kern_center(bins, D, db, dx):
    """kern_center(bins, D, db, dx)

    Calculates the number of elements within a bin of width db
        when `bins` indicates the center of the bin, and we
        go from bin-db/2 to bin+db/2.

    Args:
        bins (np.ndarray): Bin array
        D (int): Euclidean dimension
        db (np.ndarray): Bin width size
        dx (tuple,list): Element widths for each dimension, grid size
    Returns:
        np.ndarray: The number of elements within a bin of width db
    """
    if D == 1:
        return 2.
    elif D == 2:
        return (2. * np.pi * bins * db) / np.prod(dx)**2
    elif D == 3:
        return (4./3.) * np.pi * (3. * bins**2 * db + db**3 / 4.) / np.prod(dx)**3
    else:
        raise ValueError('Not implemented for dimension %s' % D)

In [27]:
bins = np.arange(1, 1024)
D = 2
db = bins.copy()
dx = (0.5, 0.5, 0.5)

k2 = kern(bins, D, db, dx)
k2_new = hypersphere_binshell(bins, D, db, dx)

In [28]:
np.allclose(k2, k2_new)

True